# 📚 TREC-COVID Dataset Exploration

This notebook explores and explains the structure of the TREC-COVID dataset used in our information retrieval project.

## Objectives
- Load and explore the document corpus
- Analyze test queries
- Examine relevance judgments (Qrels)
- Understand the complete data structure


## 📦 1. Imports and Configuration


In [ ]:
# Required imports
import json
import os
from collections import Counter

print("✓ Imports completed")


✓ Imports terminés


## 📂 2. Path Configuration


In [ ]:
# Path to the data
base_path = "data/trec-covid/trec-covid"

# Paths to files
corpus_path = os.path.join(base_path, "corpus.jsonl")
queries_path = os.path.join(base_path, "queries.jsonl")
qrels_path = os.path.join(base_path, "qrels", "test.tsv")

# Check that files exist
if not os.path.exists(corpus_path):
    raise FileNotFoundError(f"❌ Corpus file not found: {corpus_path}\n"
                          f"Run download.py first to download the dataset")

print(f"✓ Paths configured")
print(f"  - Corpus: {corpus_path}")
print(f"  - Queries: {queries_path}")
print(f"  - Qrels: {qrels_path}")


✓ Chemins configurés
  - Corpus: data/trec-covid/trec-covid\corpus.jsonl
  - Requêtes: data/trec-covid/trec-covid\queries.jsonl
  - Qrels: data/trec-covid/trec-covid\qrels\test.tsv


## 📄 3. Corpus (Documents) Exploration

The corpus contains all scientific documents about COVID-19 in which we will search.


### 3.1. Corpus Loading


In [ ]:
# Load corpus from JSONL file
# Format: each line = 1 JSON document
corpus = []
print("Loading corpus...")
with open(corpus_path, 'r', encoding='utf-8') as f:
    for line in f:
        corpus.append(json.loads(line))

print(f"✓ Corpus loaded: {len(corpus):,} documents")


Chargement du corpus...
✓ Corpus chargé: 171,332 documents


### 3.2. Example Document


In [ ]:
# Display an example document to understand the structure
doc = corpus[0]
print("📄 Example document:")
print(f"   ID: {doc['_id']}")
print(f"   Title: {doc['title'][:100]}...")
print(f"\n   Text (first 300 characters):")
print(f"   {doc['text'][:300]}...")


📄 Exemple de document:
   ID: ug7v899j
   Titre: Clinical features of culture-proven Mycoplasma pneumoniae infections at King Abdulaziz University Ho...

   Texte (premiers 300 caractères):
   OBJECTIVE: This retrospective chart review describes the epidemiology and clinical features of 40 patients with culture-proven Mycoplasma pneumoniae infections at King Abdulaziz University Hospital, Jeddah, Saudi Arabia. METHODS: Patients with positive M. pneumoniae cultures from respiratory specime...


### 3.3. Document Statistics


In [ ]:
# Calculate statistics on documents
# (We use a sample of 1000 documents for speed)
sample_size = min(1000, len(corpus))
title_lengths = [len(doc['title']) for doc in corpus[:sample_size]]
text_lengths = [len(doc['text']) for doc in corpus[:sample_size]]

print(f"📊 Statistics (on sample of {sample_size} documents):")
print(f"   • Average title length: {sum(title_lengths)/len(title_lengths):.0f} characters")
print(f"   • Average text length: {sum(text_lengths)/len(text_lengths):.0f} characters")
print(f"   • Min text length: {min(text_lengths)} characters")
print(f"   • Max text length: {max(text_lengths)} characters")


📊 Statistiques (sur échantillon de 1000 documents):
   • Longueur moyenne des titres: 96 caractères
   • Longueur moyenne des textes: 1400 caractères
   • Longueur min du texte: 0 caractères
   • Longueur max du texte: 2828 caractères


## ❓ 4. Queries Exploration

Queries are natural language questions that our system must be able to answer.


### 4.1. Queries Loading


In [ ]:
# Load queries from JSONL file
queries = []
print("Loading queries...")
with open(queries_path, 'r', encoding='utf-8') as f:
    for line in f:
        queries.append(json.loads(line))

print(f"✓ Queries loaded: {len(queries)} queries")


Chargement des requêtes...
✓ Requêtes chargées: 50 requêtes


### 4.2. Query Examples


In [ ]:
# Display some query examples
print("❓ Query examples:\n")
for i, query in enumerate(queries[:5], 1):
    print(f"   Query {i} (ID: {query['_id']}):")
    print(f"   \"{query['text']}\"")
    print()


❓ Exemples de requêtes:

   Requête 1 (ID: 1):
   "what is the origin of COVID-19"

   Requête 2 (ID: 2):
   "how does the coronavirus respond to changes in the weather"

   Requête 3 (ID: 3):
   "will SARS-CoV2 infected people develop immunity? Is cross protection possible?"

   Requête 4 (ID: 4):
   "what causes death from Covid-19?"

   Requête 5 (ID: 5):
   "what drugs have been active against SARS-CoV or SARS-CoV-2 in animal studies?"



## ✅ 5. Qrels (Relevance Judgments) Exploration

Qrels are the "Ground Truth": experts have evaluated which documents are relevant for each query. This is our reference for evaluating our model.


### 5.1. Qrels Loading


In [ ]:
# Load Qrels from TSV file
# TSV format: query_id | doc_id | relevance_score
qrels = []
print("Loading Qrels...")
with open(qrels_path, 'r', encoding='utf-8') as f:
    header = f.readline()  # Read header
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            qrels.append({
                'query_id': parts[0],
                'doc_id': parts[1],
                'relevance': int(parts[2])
            })

print(f"✓ Qrels loaded: {len(qrels):,} relevance judgments")


Chargement des Qrels...
✓ Qrels chargés: 66,336 jugements de pertinence


### 5.2. Relevance Score Distribution

**⚠️ Important:** The TREC-COVID dataset uses a simplified relevance scale.

Relevance scores in this dataset generally range from **-1 to 2**:
- **-1** = Document not evaluated or removed (rare)
- **0** = Not relevant at all
- **1** = Slightly relevant
- **2** = Relevant

**Note:** Unlike other TREC datasets that use a 0-4 scale, TREC-COVID uses this simplified scale to facilitate rapid evaluation during the pandemic.


In [ ]:
# Analyze the distribution of relevance scores
relevance_scores = [qrel['relevance'] for qrel in qrels]
relevance_counter = Counter(relevance_scores)

print("📊 Relevance score distribution:")
for score in sorted(relevance_counter.keys()):
    count = relevance_counter[score]
    percentage = (count / len(qrels)) * 100
    print(f"   • Score {score}: {count:,} judgments ({percentage:.1f}%)")


📊 Distribution des scores de pertinence:
   • Score -1: 2 jugements (0.0%)
   • Score 0: 41,661 jugements (62.8%)
   • Score 1: 10,456 jugements (15.8%)
   • Score 2: 14,217 jugements (21.4%)


### 5.3. Relevant Documents per Query


In [ ]:
# Count the number of relevant documents per query
docs_per_query = Counter([qrel['query_id'] for qrel in qrels])

print("📊 Relevant documents per query:")
print(f"   • Average: {sum(docs_per_query.values())/len(docs_per_query):.1f} relevant documents")
print(f"   • Min: {min(docs_per_query.values())} documents")
print(f"   • Max: {max(docs_per_query.values())} documents")
print(f"   • Total queries with judgments: {len(docs_per_query)}")


📊 Documents pertinents par requête:
   • Moyenne: 1326.7 documents pertinents
   • Min: 631 documents
   • Max: 1941 documents
   • Total requêtes avec jugements: 50


### 5.4. Relevance Judgment Examples


In [ ]:
# Display some relevance judgment examples
print("📋 Relevance judgment examples:")
print("   Format: query_id | doc_id | relevance_score\n")
for qrel in qrels[:5]:
    print(f"   {qrel['query_id']} | {qrel['doc_id']} | {qrel['relevance']}")


📋 Exemples de jugements de pertinence:
   Format: query_id | doc_id | score_de_pertinence

   1 | 005b2j4b | 2
   1 | 00fmeepz | 1
   1 | g7dhmyyo | 2
   1 | 0194oljo | 1
   1 | 021q9884 | 1


## 📝 6. Data Structure Summary


In [ ]:
print("=" * 80)
print("📝 DATA STRUCTURE SUMMARY")
print("=" * 80)
print("""
The TREC-COVID dataset contains 3 types of files:

1. corpus.jsonl (Documents)
   → Each line = 1 JSON document with:
      • _id: unique document identifier
      • title: scientific article title
      • text: article abstract/content

2. queries.jsonl (Queries)
   → Each line = 1 JSON query with:
      • _id: unique query identifier
      • text: natural language question (e.g., "loss of taste")

3. qrels/test.tsv (Relevance judgments)
   → TSV format with columns:
      • query_id: query ID
      • doc_id: document ID
      • relevance: relevance score (-1 to 2 in this dataset)
         - -1: Document not evaluated (rare)
         - 0: Not relevant
         - 1: Slightly relevant
         - 2: Relevant
   
   ⚠️ Note: TREC-COVID uses a simplified scale (0-2) unlike 
   other TREC datasets that use 0-4. This is normal for this dataset!
   
   These judgments are the "Ground Truth" validated by experts.
   They allow you to evaluate whether your model finds the right documents!

🎯 Objective of your project:
   For each query, your model must retrieve relevant documents
   (those with a high relevance score in the qrels, i.e., score 2).
""")


📝 RÉSUMÉ DE LA STRUCTURE DES DONNÉES

Le dataset TREC-COVID contient 3 types de fichiers:

1. corpus.jsonl (Documents)
   → Chaque ligne = 1 document JSON avec:
      • _id: identifiant unique du document
      • title: titre de l'article scientifique
      • text: résumé/contenu de l'article

2. queries.jsonl (Requêtes)
   → Chaque ligne = 1 requête JSON avec:
      • _id: identifiant unique de la requête
      • text: question en langage naturel (ex: "perte de goût")

3. qrels/test.tsv (Jugements de pertinence)
   → Format TSV avec colonnes:
      • query_id: ID de la requête
      • doc_id: ID du document
      • relevance: score de pertinence (0-4, où 4 = très pertinent)
   
   Ces jugements sont la "Vérité Terrain" validée par des experts.
   Ils permettent d'évaluer si votre modèle trouve les bons documents!

🎯 Objectif de votre projet:
   Pour chaque requête, votre modèle doit retrouver les documents pertinents
   (ceux qui ont un score de pertinence élevé dans les qrels).



## ✅ Conclusion

We have successfully explored the TREC-COVID dataset:

- ✅ Corpus loaded and analyzed
- ✅ Queries examined
- ✅ Qrels loaded and statistics calculated
- ✅ Data structure understood

**Next steps:**
- Implement TF-IDF for search
- Compare with Word2Vec
- Evaluate performance with NDCG@10
